# FinBERT V3 — Multitask Training & Evaluation

This notebook documents the current FinBERT training workflow used during Aroogula's model-development process.

The model shares a FinBERT encoder across two tasks:

1. **Sentiment** — positive / negative / neutral.
2. **Tradeability** — filter ordinary news from potential trading catalysts.

The notebook keeps train, validation and test splits separate, calibrates the tradeability threshold on validation data, evaluates once on the held-out test set, and saves deployment metadata with the trained model.

### Recorded experiment

The retained outputs in this public notebook come from the original experiment on **12,974 labeled examples**:

- Train: **10,368**
- Validation: **1,352**
- Test: **1,254**
- Test sentiment accuracy: **0.9242**
- Test sentiment macro-F1: **0.8918**
- Test tradeability F1 at the default argmax decision: **0.8331**
- Validation-calibrated tradeability threshold: **0.1421**
- Test tradeability recall at the calibrated threshold: **0.9141**

These metrics describe this specific development dataset and split. They are not trading-return metrics and should not be interpreted as evidence of strategy profitability.

## 0. Installation

Run this cell only if your environment does not already have the required dependencies. Restart the kernel afterward.


In [ ]:
# Uncomment if you need to install or update dependencies.
# %pip install -U "transformers>=4.46" accelerate safetensors pandas openpyxl scikit-learn matplotlib


## 1. Imports, Configuration, and Seeds


In [ ]:
from __future__ import annotations

import inspect
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

from torch import nn
from torch.utils.data import Dataset

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertModel,
    BertPreTrainedModel,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

DATA_PATH = Path("../data/training/FINBERT_V3_TRAINING_DATA.xlsx")
SHEET_NAME = "TRAIN_V3"

OUTPUT_DIR = Path("../artifacts/finbert_v3_runs")
BEST_MODEL_DIR = Path("../artifacts/finbert_v3_best_model")

MODEL_NAME = "ProsusAI/finbert"
MAX_LENGTH = 256

SEED = 42
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_RATIO = 0.06

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1

LAMBDA_TRADEABILITY = 1.0
USE_CLASS_WEIGHTS = False
TARGET_TRADEABLE_RECALL = 0.93
USE_TENSORBOARD = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Data file:", DATA_PATH.resolve())

## 2. Load and Validate the Dataset


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place FINBERT_V3_TRAINING_DATA.xlsx under "
        "data/training/ or update DATA_PATH."
    )

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

required_columns = {
    "text",
    "finbert_sentiment",
    "tradeable",
    "sentiment_weight",
    "tradeability_weight",
    "split",
}

missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(
        "Missing required columns: " + ", ".join(sorted(missing_columns))
    )

df["text"] = df["text"].fillna("").astype(str).str.strip()
df["finbert_sentiment"] = pd.to_numeric(
    df["finbert_sentiment"], errors="raise"
).astype(int)
df["tradeable"] = pd.to_numeric(
    df["tradeable"], errors="raise"
).astype(int)
df["sentiment_weight"] = pd.to_numeric(
    df["sentiment_weight"], errors="raise"
).astype(float)
df["tradeability_weight"] = pd.to_numeric(
    df["tradeability_weight"], errors="raise"
).astype(float)
df["split"] = df["split"].astype(str).str.lower().str.strip()

if (df["text"].str.len() == 0).any():
    raise ValueError("The dataset contains empty text entries.")
if not set(df["finbert_sentiment"].unique()).issubset({0, 1, 2}):
    raise ValueError("finbert_sentiment may only contain 0, 1, or 2.")
if not set(df["tradeable"].unique()).issubset({0, 1}):
    raise ValueError("tradeable may only contain 0 or 1.")
if (df["sentiment_weight"] <= 0).any():
    raise ValueError("All sentiment_weight values must be greater than zero.")
if (df["tradeability_weight"] <= 0).any():
    raise ValueError("All tradeability_weight values must be greater than zero.")

valid_splits = {"train", "validation", "test"}
if not set(df["split"].unique()).issubset(valid_splits):
    raise ValueError(
        f"split contains unexpected values: {sorted(df['split'].unique())}"
    )

print(f"Total rows: {len(df):,}")
display(df.head(3))


Total rows: 12,974


## 3. Use the Excel Splits

Do not reshuffle and split the entire dataset again. The Excel splits keep repeated or related news items together.


In [ ]:
train_df = df[df["split"] == "train"].reset_index(drop=True)
validation_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train:", f"{len(train_df):,}")
print("Validation:", f"{len(validation_df):,}")
print("Test:", f"{len(test_df):,}")

if min(len(train_df), len(validation_df), len(test_df)) == 0:
    raise ValueError("One of the dataset splits is empty.")


def show_distribution(dataframe: pd.DataFrame, name: str) -> None:
    sentiment_counts = (
        dataframe["finbert_sentiment"]
        .value_counts()
        .sort_index()
        .rename(index={0: "positive", 1: "negative", 2: "neutral"})
    )
    tradeable_counts = (
        dataframe["tradeable"]
        .value_counts()
        .sort_index()
        .rename(index={0: "filter_out", 1: "catalyst"})
    )

    print(f"\n{name}")
    print("Sentiment:")
    display(sentiment_counts.to_frame("count").T)
    print("Tradeability:")
    display(tradeable_counts.to_frame("count").T)


show_distribution(train_df, "TRAIN")
show_distribution(validation_df, "VALIDATION")
show_distribution(test_df, "TEST")


## 4. Tokenizer and Custom PyTorch Dataset


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class FinBertDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 256):
        self.texts = dataframe["text"].fillna("").astype(str).tolist()
        self.sentiment = dataframe["finbert_sentiment"].to_numpy(dtype=np.int64)
        self.tradeable = dataframe["tradeable"].to_numpy(dtype=np.int64)
        self.sentiment_weights = dataframe["sentiment_weight"].to_numpy(
            dtype=np.float32
        )
        self.tradeability_weights = dataframe["tradeability_weight"].to_numpy(
            dtype=np.float32
        )
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.sentiment[idx], dtype=torch.long),
            "tradeable": torch.tensor(self.tradeable[idx], dtype=torch.long),
            "sentiment_weight": torch.tensor(
                self.sentiment_weights[idx], dtype=torch.float
            ),
            "tradeability_weight": torch.tensor(
                self.tradeability_weights[idx], dtype=torch.float
            ),
        }


train_dataset = FinBertDataset(train_df, tokenizer, MAX_LENGTH)
validation_dataset = FinBertDataset(validation_df, tokenizer, MAX_LENGTH)
test_dataset = FinBertDataset(test_df, tokenizer, MAX_LENGTH)

sample = train_dataset[0]
print({key: tuple(value.shape) for key, value in sample.items()})
print("Sentiment weight:", sample["sentiment_weight"].item())
print("Tradeability weight:", sample["tradeability_weight"].item())

assert sample["input_ids"].shape == (MAX_LENGTH,)
assert sample["attention_mask"].shape == (MAX_LENGTH,)
assert sample["labels"].ndim == 0
assert sample["tradeable"].ndim == 0

## 5. Multitask Model

- Keeps the FinBERT encoder.
- Copies the original sentiment classification head.
- Creates a new classification head for `tradeable`.
- Inherits from `BertPreTrainedModel`, so it can be saved and reloaded with `save_pretrained()` and `from_pretrained()`.


In [ ]:
class MultiTaskFinBert(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)

        self.bert = BertModel(config)
        classifier_dropout = getattr(config, "classifier_dropout", None)
        if classifier_dropout is None:
            classifier_dropout = config.hidden_dropout_prob

        self.dropout = nn.Dropout(classifier_dropout)
        self.sentiment = nn.Linear(config.hidden_size, 3)
        self.tradeable = nn.Linear(config.hidden_size, 2)
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        **kwargs,
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )
        pooled_output = self.dropout(outputs.pooler_output)
        sentiment_logits = self.sentiment(pooled_output)
        tradeability_logits = self.tradeable(pooled_output)
        return sentiment_logits, tradeability_logits


def build_fresh_multitask_model() -> MultiTaskFinBert:
    original_finbert = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME
    )

    config = original_finbert.config
    config.architectures = ["MultiTaskFinBert"]
    config.num_sentiment_labels = 3
    config.num_tradeability_labels = 2

    multitask_model = MultiTaskFinBert(config)
    multitask_model.bert.load_state_dict(original_finbert.bert.state_dict())
    multitask_model.sentiment.load_state_dict(
        original_finbert.classifier.state_dict()
    )

    del original_finbert
    return multitask_model


model = build_fresh_multitask_model()

with torch.no_grad():
    model.eval()
    test_output = model(
        input_ids=sample["input_ids"].unsqueeze(0),
        attention_mask=sample["attention_mask"].unsqueeze(0),
    )

print("Sentiment logits shape:", test_output[0].shape)
print("Tradeability logits shape:", test_output[1].shape)
assert test_output[0].shape == (1, 3)
assert test_output[1].shape == (1, 2)

## 6. Optional Class Weights

Start with `USE_CLASS_WEIGHTS = False`. The weights in the Excel file already represent confidence/quality for each example. You can later repeat the experiment with class weights enabled and compare the results.


In [ ]:
def make_class_weights(
    dataframe: pd.DataFrame,
    column: str,
    classes: np.ndarray,
) -> torch.Tensor:
    values = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=dataframe[column].to_numpy(),
    )
    return torch.tensor(values, dtype=torch.float)


if USE_CLASS_WEIGHTS:
    sentiment_class_weights = make_class_weights(
        train_df,
        column="finbert_sentiment",
        classes=np.array([0, 1, 2]),
    )
    tradeability_class_weights = make_class_weights(
        train_df,
        column="tradeable",
        classes=np.array([0, 1]),
    )
else:
    sentiment_class_weights = None
    tradeability_class_weights = None

print("Sentiment class weights:", sentiment_class_weights)
print("Tradeability class weights:", tradeability_class_weights)

## 7. Trainer with Weighted Multitask Loss

The objective is:

`total_loss = sentiment_loss + lambda_tradeability * tradeability_loss`

Each task uses only its own sample weight.


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(
        self,
        lambda_tradeability: float = 1.0,
        sentiment_class_weights: torch.Tensor | None = None,
        tradeability_class_weights: torch.Tensor | None = None,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.lambda_tradeability = float(lambda_tradeability)
        self.sentiment_class_weights = sentiment_class_weights
        self.tradeability_class_weights = tradeability_class_weights

    @staticmethod
    def normalized_weighted_loss(
        losses: torch.Tensor,
        labels: torch.Tensor,
        sample_weights: torch.Tensor | None,
        class_weights: torch.Tensor | None,
    ) -> torch.Tensor:
        if sample_weights is None:
            sample_weights = torch.ones_like(
                losses,
                dtype=losses.dtype,
                device=losses.device,
            )
        else:
            sample_weights = sample_weights.to(
                device=losses.device,
                dtype=losses.dtype,
            )

        numerator = (losses * sample_weights).sum()

        if class_weights is None:
            denominator = sample_weights.sum()
        else:
            class_weights = class_weights.to(
                device=losses.device,
                dtype=losses.dtype,
            )
            effective_class_weights = class_weights[labels]
            denominator = (sample_weights * effective_class_weights).sum()

        return numerator / denominator.clamp_min(1e-8)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        labels_sentiment = inputs.pop("labels")
        labels_tradeability = inputs.pop("tradeable")

        sentiment_sample_weights = inputs.pop("sentiment_weight", None)
        tradeability_sample_weights = inputs.pop("tradeability_weight", None)

        sentiment_logits, tradeability_logits = model(**inputs)

        sentiment_class_weights = self.sentiment_class_weights
        tradeability_class_weights = self.tradeability_class_weights

        if sentiment_class_weights is not None:
            sentiment_class_weights = sentiment_class_weights.to(
                sentiment_logits.device
            )
        if tradeability_class_weights is not None:
            tradeability_class_weights = tradeability_class_weights.to(
                tradeability_logits.device
            )

        sentiment_loss_function = nn.CrossEntropyLoss(
            weight=sentiment_class_weights,
            reduction="none",
        )
        tradeability_loss_function = nn.CrossEntropyLoss(
            weight=tradeability_class_weights,
            reduction="none",
        )

        sentiment_losses = sentiment_loss_function(
            sentiment_logits,
            labels_sentiment,
        )
        tradeability_losses = tradeability_loss_function(
            tradeability_logits,
            labels_tradeability,
        )

        sentiment_loss = self.normalized_weighted_loss(
            losses=sentiment_losses,
            labels=labels_sentiment,
            sample_weights=sentiment_sample_weights,
            class_weights=sentiment_class_weights,
        )
        tradeability_loss = self.normalized_weighted_loss(
            losses=tradeability_losses,
            labels=labels_tradeability,
            sample_weights=tradeability_sample_weights,
            class_weights=tradeability_class_weights,
        )

        total_loss = (
            sentiment_loss
            + self.lambda_tradeability * tradeability_loss
        )

        outputs = {
            "sentiment_logits": sentiment_logits,
            "tradeability_logits": tradeability_logits,
        }

        return (total_loss, outputs) if return_outputs else total_loss

## 8. Metrics


In [ ]:
def compute_metrics(eval_pred) -> dict[str, float]:
    sentiment_logits, tradeability_logits = eval_pred.predictions
    sentiment_labels, tradeability_labels = eval_pred.label_ids

    sentiment_predictions = np.argmax(sentiment_logits, axis=-1)
    tradeability_predictions = np.argmax(tradeability_logits, axis=-1)

    sentiment_class_f1 = f1_score(
        sentiment_labels,
        sentiment_predictions,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )
    sentiment_macro_f1 = f1_score(
        sentiment_labels,
        sentiment_predictions,
        average="macro",
        zero_division=0,
    )
    tradeability_f1 = f1_score(
        tradeability_labels,
        tradeability_predictions,
        zero_division=0,
    )

    return {
        "sentiment_accuracy": accuracy_score(
            sentiment_labels,
            sentiment_predictions,
        ),
        "sentiment_macro_f1": sentiment_macro_f1,
        "sentiment_positive_f1": sentiment_class_f1[0],
        "sentiment_negative_f1": sentiment_class_f1[1],
        "sentiment_neutral_f1": sentiment_class_f1[2],
        "tradeable_accuracy": accuracy_score(
            tradeability_labels,
            tradeability_predictions,
        ),
        "tradeable_precision": precision_score(
            tradeability_labels,
            tradeability_predictions,
            zero_division=0,
        ),
        "tradeable_recall": recall_score(
            tradeability_labels,
            tradeability_predictions,
            zero_division=0,
        ),
        "tradeable_f1": tradeability_f1,
        "combined_score": (sentiment_macro_f1 + tradeability_f1) / 2.0,
    }

## 9. TrainingArguments

In [ ]:
training_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "combined_score",
    "greater_is_better": True,
    "label_names": ["labels", "tradeable"],
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_train_epochs": NUM_EPOCHS,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": "linear",
    "max_grad_norm": 1.0,
    "remove_unused_columns": False,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "logging_steps": 50,
    "logging_dir": str(OUTPUT_DIR / "logs"),
    "fp16": torch.cuda.is_available(),
    "dataloader_num_workers": 0,
    "report_to": ["tensorboard"] if USE_TENSORBOARD else [],
    "seed": SEED,
    "data_seed": SEED,
}

training_argument_parameters = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in training_argument_parameters:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)
print(training_args)

## 10. Create the Trainer


In [ ]:
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": validation_dataset,
    "compute_metrics": compute_metrics,
    "callbacks": [
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.001,
        )
    ],
    "lambda_tradeability": LAMBDA_TRADEABILITY,
    "sentiment_class_weights": sentiment_class_weights,
    "tradeability_class_weights": tradeability_class_weights,
}

try:
    trainer = WeightedTrainer(
        processing_class=tokenizer,
        **trainer_kwargs,
    )
except TypeError:
    trainer = WeightedTrainer(
        tokenizer=tokenizer,
        **trainer_kwargs,
    )

print("Trainer ready.")


## 11. Train

Before running this section, confirm that the train, validation, and test split sizes are correct. The test split is not used for model selection.


In [ ]:
train_result = trainer.train()
print("Training complete.")
print(train_result.metrics)


Training complete.
{'train_runtime': 423.8709, 'train_loss': 0.6167429008601625, 'epoch': 3.0}


## 12. Visualize Training History


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
display(history.tail(10))

if "loss" in history.columns:
    train_history = history.dropna(subset=["loss"])

    plt.figure(figsize=(8, 4))
    plt.plot(
        train_history["epoch"],
        train_history["loss"],
        marker="o",
        label="train loss",
    )

    if "eval_loss" in history.columns:
        eval_history = history.dropna(subset=["eval_loss"])
        plt.plot(
            eval_history["epoch"],
            eval_history["eval_loss"],
            marker="o",
            label="validation loss",
        )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

## 13. Evaluation

The validation split is used for model selection and calibration. The test split is evaluated only once at the end.


In [ ]:
validation_results = trainer.evaluate(
    eval_dataset=validation_dataset,
    metric_key_prefix="validation",
)

test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

print("VALIDATION")
display(pd.Series(validation_results).to_frame("value"))

print("\nFINAL TEST")
display(pd.Series(test_results).to_frame("value"))


VALIDATION
sentiment_accuracy=0.9061 | sentiment_macro_f1=0.8690 | tradeable_precision=0.8000 | tradeable_recall=0.8617 | tradeable_f1=0.8297 | combined_score=0.8493

FINAL TEST
sentiment_accuracy=0.9242 | sentiment_macro_f1=0.8918 | tradeable_precision=0.8179 | tradeable_recall=0.8488 | tradeable_f1=0.8331 | combined_score=0.8624


## 14. Calibrate the `tradeable` Threshold


In [ ]:
def softmax_numpy(logits: np.ndarray) -> np.ndarray:
    tensor = torch.tensor(logits, dtype=torch.float32)
    return torch.softmax(tensor, dim=-1).numpy()


validation_prediction_output = trainer.predict(validation_dataset)
validation_sentiment_logits, validation_tradeability_logits = (
    validation_prediction_output.predictions
)
validation_sentiment_labels, validation_tradeability_labels = (
    validation_prediction_output.label_ids
)

validation_tradeability_probabilities = softmax_numpy(
    validation_tradeability_logits
)[:, 1]

precision_values, recall_values, thresholds = precision_recall_curve(
    validation_tradeability_labels,
    validation_tradeability_probabilities,
)

candidate_indices = np.where(
    recall_values[:-1] >= TARGET_TRADEABLE_RECALL
)[0]

if len(candidate_indices) == 0:
    TRADEABLE_THRESHOLD = 0.50
    print(
        "No threshold reached the target recall. "
        "Falling back to 0.50."
    )
else:
    best_candidate = candidate_indices[
        np.argmax(precision_values[:-1][candidate_indices])
    ]
    TRADEABLE_THRESHOLD = float(thresholds[best_candidate])

validation_tradeability_predictions = (
    validation_tradeability_probabilities >= TRADEABLE_THRESHOLD
).astype(int)

print("Tradeable threshold:", round(TRADEABLE_THRESHOLD, 4))
print(
    "Validation precision:",
    round(
        precision_score(
            validation_tradeability_labels,
            validation_tradeability_predictions,
            zero_division=0,
        ),
        4,
    ),
)
print(
    "Validation recall:",
    round(
        recall_score(
            validation_tradeability_labels,
            validation_tradeability_predictions,
            zero_division=0,
        ),
        4,
    ),
)
print(
    "Validation F1:",
    round(
        f1_score(
            validation_tradeability_labels,
            validation_tradeability_predictions,
            zero_division=0,
        ),
        4,
    ),
)


Tradeable threshold: 0.1421
Validation precision: 0.7022
Validation recall: 0.9325
Validation F1: 0.8011


## 15. Detailed Reports


In [ ]:
SENTIMENT_NAMES = ["positive", "negative", "neutral"]
TRADEABILITY_NAMES = ["filter_out", "catalyst"]


def evaluate_detailed(dataset, dataset_name: str) -> None:
    output = trainer.predict(dataset)

    sentiment_logits, tradeability_logits = output.predictions
    sentiment_labels, tradeability_labels = output.label_ids

    sentiment_predictions = np.argmax(sentiment_logits, axis=-1)
    tradeability_probabilities = softmax_numpy(tradeability_logits)[:, 1]
    tradeability_predictions = (
        tradeability_probabilities >= TRADEABLE_THRESHOLD
    ).astype(int)

    print(f"\n===== {dataset_name}: SENTIMENT =====")
    print(
        classification_report(
            sentiment_labels,
            sentiment_predictions,
            labels=[0, 1, 2],
            target_names=SENTIMENT_NAMES,
            digits=4,
            zero_division=0,
        )
    )
    print(
        confusion_matrix(
            sentiment_labels,
            sentiment_predictions,
            labels=[0, 1, 2],
        )
    )

    print(f"\n===== {dataset_name}: TRADEABILITY =====")
    print(
        classification_report(
            tradeability_labels,
            tradeability_predictions,
            labels=[0, 1],
            target_names=TRADEABILITY_NAMES,
            digits=4,
            zero_division=0,
        )
    )
    print(
        confusion_matrix(
            tradeability_labels,
            tradeability_predictions,
            labels=[0, 1],
        )
    )


evaluate_detailed(validation_dataset, "VALIDATION")
evaluate_detailed(test_dataset, "FINAL TEST")


VALIDATION — calibrated tradeability threshold
Sentiment: accuracy=0.9061, macro-F1=0.8690
Tradeability: precision=0.7022, recall=0.9325, F1=0.8011

FINAL TEST — calibrated tradeability threshold
Sentiment: accuracy=0.9242, macro-F1=0.8918
Tradeability: precision=0.6803, recall=0.9141, F1=0.7801


## 16. Save the Model and Deployment Configuration

The model is saved with `save_pretrained()`. There is no longer any need to rename keys or use `strict=False`.


In [ ]:
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

deployment_config = {
    "base_model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "sentiment_labels": {
        "0": "positive",
        "1": "negative",
        "2": "neutral",
    },
    "tradeability_labels": {
        "0": False,
        "1": True,
    },
    "tradeable_threshold": TRADEABLE_THRESHOLD,
    "target_tradeable_recall": TARGET_TRADEABLE_RECALL,
    "input_format": (
        "Target company: {company_name} ({ticker}). "
        "Headline: {title} Summary: {summary}"
    ),
    "lambda_tradeability": LAMBDA_TRADEABILITY,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "seed": SEED,
}

with open(
    BEST_MODEL_DIR / "deployment_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(deployment_config, file, indent=2, ensure_ascii=False)

print("Model saved to:", BEST_MODEL_DIR.resolve())
for path in sorted(BEST_MODEL_DIR.iterdir()):
    print("-", path.name)


## 17. Test Model Reloading


In [ ]:
reloaded_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
reloaded_model = MultiTaskFinBert.from_pretrained(BEST_MODEL_DIR)

reloaded_model.to(DEVICE)
reloaded_model.eval()

print("Model reloaded successfully.")


## 18. Inference Function

Use exactly the same text format that was used in the training dataset.


In [ ]:
def build_model_text(
    ticker: str,
    title: str,
    summary: str,
    company_name: str | None = None,
) -> str:
    company_name = company_name or ticker
    return (
        f"Target company: {company_name} ({ticker}). "
        f"Headline: {str(title).strip()} "
        f"Summary: {str(summary).strip()}"
    )


def predict_finbert_v3(
    ticker: str,
    title: str,
    summary: str,
    company_name: str | None = None,
) -> dict:
    text = build_model_text(
        ticker=ticker,
        title=title,
        summary=summary,
        company_name=company_name,
    )

    inputs = reloaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        sentiment_logits, tradeability_logits = reloaded_model(**inputs)

    sentiment_probabilities = torch.softmax(sentiment_logits, dim=-1)[0]
    tradeability_probabilities = torch.softmax(tradeability_logits, dim=-1)[0]

    sentiment_index = int(torch.argmax(sentiment_probabilities).item())
    tradeable_probability = float(tradeability_probabilities[1].item())

    return {
        "sentiment": SENTIMENT_NAMES[sentiment_index],
        "sentiment_confidence": float(
            sentiment_probabilities[sentiment_index].item()
        ),
        "tradeable": tradeable_probability >= TRADEABLE_THRESHOLD,
        "tradeable_probability": tradeable_probability,
        "tradeable_threshold": TRADEABLE_THRESHOLD,
        "text": text,
    }


example = predict_finbert_v3(
    ticker="AAPL",
    company_name="Apple Inc.",
    title="Apple raises its revenue outlook",
    summary="The company increased guidance after stronger-than-expected demand.",
)

example

## 19. Required Changes in `TradeAnalyzer`

Build the input using the same format:

```python
text = build_model_text(
    ticker=ticker,
    company_name=company_name,
    title=title,
    summary=summary,
)
```

Load the model as follows:

```python
self.tokenizer = AutoTokenizer.from_pretrained(model_path)
self.model = MultiTaskFinBert.from_pretrained(model_path)
self.model.to(self.finbert_device)
self.model.eval()
```

There is no longer any need for `load_file`, renaming `state_dict` keys, or using `strict=False`.

For `tradeable`, use the probability of class `1` together with the threshold stored in `deployment_config.json`.


## 20. Final Checklist

- [ ] `sentiment_macro_f1` is reasonable on both validation and test data.
- [ ] `sentiment_negative_f1` has not collapsed.
- [ ] `tradeable_recall` approximately reaches the target.
- [ ] `tradeable` precision is not unacceptably low.
- [ ] Validation and test results are reasonably similar.
- [ ] The model reloads successfully.
- [ ] `TradeAnalyzer` builds the exact same input text format.
- [ ] The threshold from `deployment_config.json` is used in production.


## Repository note

This notebook is intentionally limited to model training, evaluation and deployment preparation. Runtime trading orchestration, broker execution, portfolio state and risk management live in the main Aroogula backend.

Generated model checkpoints and training runs should remain outside Git history (for example under `artifacts/`) unless a specific release requires them.